In [10]:
import pandas as pd

In [ ]:
path = r'C:\Yoonsuk_PC\Google Drive\jayjang1194\2. Research\Toward-Business-Intelligence\data\\'
file = 'merged_df.parquet'

In [7]:
df =  pd.read_parquet(path + file)

In [8]:
df

,t_dat,customer_id,article_id,price,sales_channel_id,FN,Active,club_member_status,fashion_news_frequency,age
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2,False,False,ACTIVE,NONE,24
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2,False,False,ACTIVE,NONE,24
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2,True,True,ACTIVE,Regularly,32
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2,True,True,ACTIVE,Regularly,32
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2,True,True,ACTIVE,Regularly,32
...,...,...,...,...,...,...,...,...,...,...
31788319,2020-09-22,fff2282977442e327b45d8c89afde25617d00124d0f999...,929511001,0.059305,2,True,True,ACTIVE,Regularly,32
31788320,2020-09-22,fff2282977442e327b45d8c89afde25617d00124d0f999...,891322004,0.042356,2,True,True,ACTIVE,Regularly,32
31788321,2020-09-22,fff380805474b287b05cb2a7507b9a013482f7dd0bce0e...,918325001,0.043203,1,False,False,ACTIVE,NONE,67
31788322,2020-09-22,fff4d3a8b1f3b60af93e78c30a7cb4cf75edaf2590d3e5...,833459002,0.006763,1,True,True,ACTIVE,Regularly,21


In [9]:

# 고객 단위 집계
customer_table = df.groupby('customer_id').agg(
    age=('age', 'first'),                    # 고객 속성: 아무 행이나 대표값
    club_member_status=('club_member_status', 'first'),
    fashion_news_frequency=('fashion_news_frequency', 'first'),
    last_purchase=('t_dat', 'max'),             # R, Recency: 마지막 구매 일자
    purchase_count=('t_dat', 'count'),          # F, Frequency: 구매 횟수
    total_spent=('price', 'sum'),               # M, Monetary: 구매 금액
).reset_index()

# 최근성 계산
customer_table['last_purchase'] = pd.to_datetime(customer_table['last_purchase']) # 마지막 거래 날짜
ref_date = customer_table['last_purchase'].max() # 모든 고객 중 가장 최근에 일어난 거래 날짜
customer_table['days_since_last_purchase'] = (ref_date - customer_table['last_purchase']).dt.days # ref_date에서 구매가 얼마동안 안일어났는지

# 저장
# customer_table.to_parquet('data/customers_dashboard.parquet', index=False)
print(customer_table.shape)      # (고객수, 열수)
print(customer_table.head())

(1362281, 8)
                                         customer_id  age club_member_status  \
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...   49             ACTIVE   
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...   25             ACTIVE   
2  000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...   24             ACTIVE   
3  00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...   54             ACTIVE   
4  00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...   52             ACTIVE   

  fashion_news_frequency last_purchase  purchase_count  total_spent  \
0                   NONE    2020-09-05              21     0.648983   
1                   NONE    2020-07-08              86     2.601932   
2                   NONE    2020-09-15              18     0.704780   
3                   NONE    2019-06-09               2     0.060983   
4              Regularly    2020-08-12              13     0.469695   

   days_since_last_purchase  
0                        17  
1                  